# [8.2] Attribution Patching and EAP - Solutions

This notebook validates the helper contracts, inspects the committed CUDA signature result, and then reruns the live TransformerLens exact-vs-attribution preflight through `solutions.py`.

<details>
<summary>Expected output</summary>

All local tests should pass. The signature result should show exact scores `[0, 0, 0, 0, 0, 1]`, attribution final recovery about `0.9608`, IG final recovery about `0.9732`, EAP top edge `(5, 5)`, peak VRAM below `1 GB`, and a live CUDA run on `torch 2.12.1+cu132` with CUDA runtime `13.2`.

</details>

<details>
<summary>Help - why rerun live CUDA?</summary>

The committed report is the review artifact, but the live cell proves the current `uv` environment, GPU, TransformerLens loader, tokenizer revision, gradients, and patching hook still work together.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t

chapter = "chapter8_automated_circuits"
section = "part2_attribution_patching_eap"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_attribution_patching_eap.tests as tests
import part2_attribution_patching_eap.utils as utils

GT_TIER = "GT-1"
EXERCISE_ID = "8_2_attribution_patching_and_eap"
EXPECTED_RUNTIME = "25-45 minutes for exercises; about 1-2 minutes for the CUDA preflight"
REQUIRES_GPU = True

from part2_attribution_patching_eap import solutions


## Unit Contracts

The tests are deliberately small. They catch sign errors, component-axis reduction bugs, path-gradient rank bugs, EAP shape mistakes, invalid thresholds, non-finite values, and undocumented false negatives before the live model path.

<details>
<summary>Help - why not only test the CUDA report?</summary>

A single CUDA boolean is too coarse for debugging. If attribution patching fails, you need to know whether the bug is the first-order formula, the IG path average, the EAP matrix, the exact-vs-approx reports, or the false-negative accounting.

</details>


In [ ]:
tests.test_attribution_patch_scores_sums_non_component_dims(
    solutions.attribution_patch_scores,
)
tests.test_attribution_patch_scores_rejects_degenerate_inputs(
    solutions.attribution_patch_scores,
)
tests.test_integrated_gradient_patch_scores_average_path_gradients(
    solutions.integrated_gradient_patch_scores,
)
tests.test_integrated_gradient_patch_scores_rejects_empty_or_nonfinite_paths(
    solutions.integrated_gradient_patch_scores,
)
tests.test_edge_attribution_scores_forms_upstream_downstream_matrix(
    solutions.edge_attribution_scores,
)
tests.test_edge_attribution_scores_rejects_empty_or_nonfinite_inputs(
    solutions.edge_attribution_scores,
)
tests.test_exact_vs_approx_reports_measure_correlation_and_topk_overlap(
    solutions.score_correlation_report,
    solutions.topk_overlap_report,
)
tests.test_exact_vs_approx_reports_reject_bad_thresholds_and_scores(
    solutions.score_correlation_report,
    solutions.topk_overlap_report,
)
tests.test_runtime_and_false_negative_reports_enforce_accountability(
    solutions.runtime_improvement_report,
    solutions.false_negative_report,
)
tests.test_runtime_and_false_negative_reports_reject_nonfinite_inputs(
    solutions.runtime_improvement_report,
    solutions.false_negative_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


## CPU Contract

Before the live model, the smoke report should already have the section shape: attribution scores, IG scores, EAP edge scores, exact-vs-approx agreement, runtime speedup, and false-negative documentation.

<details>
<summary>Expected output</summary>

`attribution_scores` and `integrated_gradients` should be `[1.0, 3.0]`, `edge_scores` should be `[[3.0, 0.0], [0.0, 8.0]]`, correlation should be about `0.9966`, top-k overlap should be `1.0`, speedup should be `5.0`, and false-negative index `1` should be documented.

</details>

<details>
<summary>Common bug</summary>

A high correlation is not enough. If an exact-important component is missed by the approximation, the report must show it and require a note.

</details>


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["attribution_scores"] == [1.0, 3.0]
assert contract["integrated_gradients"] == [1.0, 3.0]
assert contract["edge_scores"] == [[3.0, 0.0], [0.0, 8.0]]
assert contract["correlation"]["passes_threshold"]
assert contract["correlation"]["correlation"] > 0.99
assert contract["topk_overlap"]["topk_overlap"] == 1.0
assert contract["runtime"]["speedup"] == 5.0
assert contract["false_negative"]["false_negative_indices"] == (1,)
assert contract["false_negative"]["documented"]
utils.print_report(
    "CPU exact-vs-approx contract",
    {
        "attribution_scores": contract["attribution_scores"],
        "integrated_gradients": contract["integrated_gradients"],
        "edge_scores": contract["edge_scores"],
        "correlation": round(contract["correlation"]["correlation"], 4),
        "topk_overlap": contract["topk_overlap"]["topk_overlap"],
        "speedup": contract["runtime"]["speedup"],
        "false_negative_indices": contract["false_negative"]["false_negative_indices"],
    },
)


## Signature Result

Now inspect the accepted CUDA report. The important result is agreement with exact patching on the top residual position, not only the final boolean.

<details>
<summary>Interpreting the signature result</summary>

Exact patching says only position `5` recovers the clean behavior. Attribution and IG are not exactly `1.0`, because they are gradient approximations, but they rank the same final position first. The EAP matrix also puts its top edge at `(5, 5)`. This is useful mechanics evidence, not a broad claim that EAP will always match exact patching.

</details>


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"] and report["tests_passed"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "gelu-1l"
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603"
assert gpu["tokenizer_revision"] == "0f6671571a20be9756b9991d978047c03b75e749"
assert gpu["hook_name"] == "blocks.0.hook_resid_post"
assert gpu["exact_patch_scores_by_position"] == [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
assert gpu["exact_best_position"] == gpu["target_position"] == 5
assert gpu["attribution_best_position"] == 5
assert gpu["ig_best_position"] == 5
assert gpu["attribution_final_recovery"] >= 0.9
assert gpu["ig_final_recovery"] >= 0.95
assert gpu["exact_attribution_top1_overlap"] == 1.0
assert gpu["exact_ig_top1_overlap"] == 1.0
assert gpu["eap_top_edge_upstream_position"] == 5
assert gpu["eap_top_edge_downstream_position"] == 5
assert gpu["peak_vram_gb"] <= 24.0

positions = list(range(gpu["sequence_length"]))
fig, (ax_scores, ax_edge) = plt.subplots(
    1,
    2,
    figsize=(9, 3.2),
    gridspec_kw={"width_ratios": [2.1, 1.1]},
)
width = 0.24
ax_scores.bar(
    [p - width for p in positions],
    gpu["exact_patch_scores_by_position"],
    width=width,
    label="exact",
)
ax_scores.bar(
    positions,
    gpu["attribution_scores_by_position"],
    width=width,
    label="attribution",
)
ax_scores.bar(
    [p + width for p in positions],
    gpu["ig_scores_by_position"],
    width=width,
    label="IG",
)
ax_scores.set_title("Exact vs approximate patch scores")
ax_scores.set_xlabel("sequence position")
ax_scores.set_ylabel("recovered fraction")
ax_scores.set_ylim(0, 1.1)
ax_scores.set_xticks(positions)
ax_scores.legend(frameon=False)

edge_heatmap = t.zeros(gpu["eap_edge_score_shape"]).float()
edge_heatmap[
    gpu["eap_top_edge_upstream_position"],
    gpu["eap_top_edge_downstream_position"],
] = gpu["eap_top_edge_score_abs"]
im = ax_edge.imshow(edge_heatmap, cmap="magma")
ax_edge.set_title("EAP top edge")
ax_edge.set_xlabel("downstream position")
ax_edge.set_ylabel("upstream position")
ax_edge.set_xticks(positions)
ax_edge.set_yticks(positions)
fig.colorbar(im, ax=ax_edge, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

utils.print_report(
    "Committed CUDA signature result",
    {
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "device": gpu["device"],
        "exact_final_recovery": gpu["exact_final_recovery"],
        "attribution_final_recovery": round(gpu["attribution_final_recovery"], 4),
        "ig_final_recovery": round(gpu["ig_final_recovery"], 4),
        "exact_attribution_correlation": round(gpu["exact_attribution_correlation"], 4),
        "exact_ig_correlation": round(gpu["exact_ig_correlation"], 4),
        "eap_top_edge": (
            gpu["eap_top_edge_upstream_position"],
            gpu["eap_top_edge_downstream_position"],
        ),
        "eap_top_edge_abs": round(gpu["eap_top_edge_score_abs"], 4),
        "peak_vram_gb": round(gpu["peak_vram_gb"], 4),
    },
)


## Live CUDA Path

The report above is committed evidence. This cell reruns the live CUDA path on the current machine through `solutions.py`.

<details>
<summary>Expected output</summary>

`preflight_passed` should be `True`, peak VRAM should stay below the 24GB budget, exact / attribution / IG best positions should all be `5`, and the EAP top edge should be `(5, 5)`.

</details>


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_full_experiment(max_vram_gb=max_vram_gb)


live_gpu = run_full_experiment(max_vram_gb=24.0)
assert live_gpu["preflight_passed"]
assert live_gpu["exact_patch_scores_by_position"] == [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
assert live_gpu["exact_best_position"] == live_gpu["target_position"] == 5
assert live_gpu["attribution_best_position"] == 5
assert live_gpu["ig_best_position"] == 5
assert live_gpu["exact_attribution_top1_overlap"] == 1.0
assert live_gpu["exact_ig_top1_overlap"] == 1.0
assert live_gpu["eap_top_edge_upstream_position"] == 5
assert live_gpu["eap_top_edge_downstream_position"] == 5
assert live_gpu["peak_vram_gb"] <= 24.0
utils.print_report(
    "Live CUDA exact-vs-attribution preflight",
    {
        "torch": live_gpu["torch_version"],
        "cuda": live_gpu["cuda_version"],
        "device": live_gpu["device"],
        "exact": live_gpu["exact_patch_scores_by_position"],
        "attribution": [round(x, 4) for x in live_gpu["attribution_scores_by_position"]],
        "ig": [round(x, 4) for x in live_gpu["ig_scores_by_position"]],
        "eap_top_edge": (
            live_gpu["eap_top_edge_upstream_position"],
            live_gpu["eap_top_edge_downstream_position"],
        ),
        "peak_vram_gb": round(live_gpu["peak_vram_gb"], 4),
    },
)


## Limitations

This is a GT-1 attribution/EAP mechanics preflight on one pinned `gelu-1l` hook and one safe prompt pair. It is not IOI-scale EAP replication, not a full path-patching intervention suite, not a dataset-level causal graph, and not evidence that attribution patching is reliable without exact-patching checks.

## Further Research

Repeat exact-vs-approx comparisons across prompt templates, move from residual positions to heads/MLPs/SAE features, compare EAP edges against true path-patching interventions, and add false-negative case studies where high global correlation still misses an exact-important component.
